# 00 - Data Pipeline

**Section 1 of the assignment: data handling and memory management.**

This notebook (a) confirms the raw dataset covers the week we need to forecast
(Dec 16-22, 2013), (b) demonstrates the memory cost of a naive full load versus
`forecasting.DataLoader`'s two-pass chunked aggregation, (c) runs the loader over
all 62 raw daily files to build the processed dataset every later notebook reads,
and (d) ranks Milan's 10,000 grid squares by total traffic to identify the three
squares used throughout the rest of the project.

All heavy logic lives in the `forecasting` package (`forecasting/data.py`); this
notebook only calls it and narrates what came back.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import time

import pandas as pd

from forecasting.data import DataLoader, naive_load_day, measure_peak_memory

RAW_DIR = ROOT / "data" / "raw"
DAILY_DIR = ROOT / "data" / "processed" / "daily"
COMBINED_PATH = ROOT / "data" / "processed" / "internet_traffic.parquet"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
raw_files = sorted(RAW_DIR.glob("sms-call-internet-mi-*.txt"))
dates = [f.stem.replace("sms-call-internet-mi-", "") for f in raw_files]
print(f"{len(raw_files)} raw files found, {dates[0]} .. {dates[-1]}")

required_week = {f"2013-12-{d:02d}" for d in range(16, 23)}
missing = required_week - set(dates)
if missing:
    raise RuntimeError(f"Missing raw files for the Dec 16-22 evaluation week: {sorted(missing)}")
print("Dec 16-22 evaluation week: all 7 daily files present.")

**Confirmed.** All 62 daily files are present, spanning 2013-11-01 through
2014-01-01, which fully covers the December 16-22 week required for the
forecasting evaluation in `02_experiments.ipynb`. Nothing needs to be
downloaded or patched before continuing.

In [ ]:
# Memory comparison: naive single read_csv vs. DataLoader's two-pass approach,
# each measured in an isolated subprocess (fair, clean-slate comparison).
sample_file = raw_files[0]
tmp_out = RESULTS_DIR / "_memory_demo.parquet"

naive_peak = measure_peak_memory(naive_load_day, sample_file)
optimized_peak = measure_peak_memory(DataLoader().process_day, sample_file, tmp_out)
tmp_out.unlink(missing_ok=True)

memory_table = pd.DataFrame([
    {"approach": "Naive (read_csv, all columns, default dtypes)", "peak_MB": naive_peak / 1e6},
    {"approach": "DataLoader (two-pass, chunked, downcast, preallocated array)", "peak_MB": optimized_peak / 1e6},
])
memory_table

**Interpretation.** The naive approach loads all 8 raw columns at their default
(64-bit) dtypes in a single `read_csv` call, so its peak working set scales
directly with file size (~322MB raw -> several hundred MB resident); run across
all 62 files at once this would not fit comfortably in memory on a laptop-class
machine. `DataLoader.process_day` bounds memory two ways at once: it only ever
holds one chunk's worth of raw rows at a time (chunked `read_csv`), and instead
of accumulating per-chunk partial DataFrames it aggregates directly into a
preallocated `(10,000 squares x ~144 timestamps)` `float32` array - a few MB,
fixed regardless of file size or chunk count. The measured gap above is the
concrete evidence for that design choice; the trade-off is a small amount of
extra bookkeeping code (the two-pass timestamp scan) in exchange for a memory
bound that no longer depends on how large the raw file is.

In [ ]:
# Build the full processed dataset: one two-pass aggregation per raw day,
# skipping any day already processed (idempotent), then a single combine.
loader = DataLoader()

t0 = time.time()
stats = loader.build_all(RAW_DIR, DAILY_DIR)
build_seconds = time.time() - t0
print(f"Processed {len(stats)} new day(s) in {build_seconds:.1f}s "
      f"({len(list(DAILY_DIR.glob('*.parquet')))} day-files total on disk).")

t0 = time.time()
combined = loader.combine(DAILY_DIR, COMBINED_PATH)
print(f"Combined into {COMBINED_PATH.name}: {combined.shape} in {time.time()-t0:.1f}s")
combined.head()

**Interpretation.** Each daily file is aggregated independently and idempotently
(re-running this cell only processes days that don't have output yet), then
`combine()` concatenates the 62 per-day Parquet files into one dataset sorted by
`(square_id, timestamp)`. The combined file is a compact, columnar `int16`/
`float32` representation of what started as ~20GB of raw tab-separated text -
small enough to read back into memory in full for the exploratory analysis in
the next notebook, and to filter efficiently per-square (via `SquareSeries`,
using Parquet predicate pushdown) throughout the modeling notebooks.

In [ ]:
totals = combined.groupby("square_id")["internet_traffic"].sum().sort_values(ascending=False)
top3 = totals.head(3)
print("Top 3 squares by total internet traffic (Nov 1 - Jan 1):")
print(top3)

top_squares_info = {
    "top3_square_ids": [int(s) for s in top3.index],
    "top3_totals": {int(s): float(v) for s, v in top3.items()},
    "fixed_reference_squares": [4159, 4556],
}
with open(RESULTS_DIR / "top_squares.json", "w") as f:
    json.dump(top_squares_info, f, indent=2)
top_squares_info

**Interpretation.** These three squares - the highest-traffic areas over the
full two-month window - are the ones used for every later analysis and
forecasting experiment (`01_eda.ipynb` onward), per the assignment's
instruction to identify and focus on the top-3-traffic areas. The result is
saved to `results/top_squares.json` so downstream notebooks read it back
instead of recomputing it, keeping "which squares are we forecasting" defined
in exactly one place.